In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.style
import seaborn as sns
import unicodedata
import difflib

In [2]:
df = pd.read_csv("../data/train.csv")

# 営業担当者のセールスした商品の種類

basic（ベーシック）: 必要最低限のサービスに絞った、最も安価なプラン。

standard（スタンダード）: 標準的な価格と内容の、一般的なプラン。

deluxe（デラックス）: スタンダードよりワンランク上の、贅沢なプラン。

superdeluxe（スーパーデラックス）: 高級ホテルなどを利用する、かなり豪華なプラン。

king（キング）: 最高級・VIP向けの、最も高額で特別な最上位プラン。

In [3]:
df["ProductPitched"]

0          Basic
1       Standard
2          Basic
3       Standard
4          Basic
          ...   
3484       Basic
3485       Basic
3486    Standard
3487        King
3488       Basic
Name: ProductPitched, Length: 3489, dtype: str

In [4]:
df["ProductPitched"].values.ravel()

<StringArray>
[       'Basic',     'Standard',        'Basic',     'Standard',
        'Basic',        'Basic', 'Super Deluxe',        'Basic',
        'basic',        'Basic',
 ...
 'SUPER DELUXE',       'Deluxe',        'Basic', 'Super Deluxe',
        'Basic',        'Basic',        'Basic',     'Standard',
         'King',        'Basic']
Length: 3489, dtype: str

In [5]:
df["ProductPitched"].value_counts()

ProductPitched
Basic           887
Deluxe          836
Standard        648
Super Deluxe    238
basic           106
               ... 
ΒASIС             1
Super 𝙳eluxe      1
Տtanda𝘳d          1
Basıϲ             1
ЅTANDARD          1
Name: count, Length: 76, dtype: int64

In [6]:
df["ProductPitched"].value_counts()

ProductPitched
Basic           887
Deluxe          836
Standard        648
Super Deluxe    238
basic           106
               ... 
ΒASIС             1
Super 𝙳eluxe      1
Տtanda𝘳d          1
Basıϲ             1
ЅTANDARD          1
Name: count, Length: 76, dtype: int64

In [7]:
correct_categories = ['basic', 'standard', 'deluxe', 'superdeluxe', 'king']
def robust_fix_product(x):
    #小文字化、空白削除、正規化
    x = str(x).replace(' ', '').lower()
    x = unicodedata.normalize('NFKC', x)
    
    #文字列が似ているものを探す（類似度30%以上で判定）
    matches = difflib.get_close_matches(x, correct_categories, n=1, cutoff=0.3)
    
    #似ているものが見つかればそれを返し、判定不能なほど壊れていれば最頻値の'basic'とする
    if matches:
        return matches[0]
    else:
        return 'basic'

df["ProductPitched"] = df["ProductPitched"].apply(robust_fix_product)

### Pythonの標準ライブラリ difflibアルゴリズムの仕組み

* 一番長く一致する部分を探す: 比較する2つの文字列（例：standard と stanᗞard）の中で、最も長く連続して同じ文字が並んでいる部分（この場合は stan）を見つける。

* 残りの部分も比較: 見つかった部分の「右側」と「左側」に残った文字同士で、さらに同じ処理を繰り返します（この場合は ard も一致）。

* スコア化: 最終的に、全体の文字数に対して「一致した文字数」がどれくらい多いかを計算し、0.0〜1.0のスコアを算出。

* このスコアが一定の基準（先ほどのコードでは cutoff=0.3、つまり30%以上）を超えていれば、「似ている」と判定して正しいカテゴリに修正する仕組みです。タイポや変な記号が1〜2文字混ざっていても、残りの部分が一致していれば同一とみなすことができる。

In [8]:
df["ProductPitched"].value_counts()

ProductPitched
basic          1157
deluxe         1045
standard        841
superdeluxe     320
king            126
Name: count, dtype: int64

In [9]:
grade_mapping = {
    'basic': 1, 
    'standard': 2, 
    'deluxe': 3, 
    'superdeluxe': 4, 
    'king': 5
}
df['ProductPitched'] = df['ProductPitched'].map(grade_mapping)

In [11]:
df["ProductPitched"].value_counts()

ProductPitched
1    1157
3    1045
2     841
4     320
5     126
Name: count, dtype: int64